### 准备数据集

In [1]:
# 加载tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('hfl/rbt3')

config.json:   0%|          | 0.00/828 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/19.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [2]:
tokenizer

BertTokenizer(name_or_path='hfl/rbt3', vocab_size=21128, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [4]:
# 试用
tokenizer(['明月装饰了你的窗子', '你装饰了别人的梦'], 
                           truncation=True)

{'input_ids': [[101, 3209, 3299, 6163, 7652, 749, 872, 4638, 4970, 2094, 102], [101, 872, 6163, 7652, 749, 1166, 782, 4638, 3457, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [5]:
# 从磁盘加载中文数据集
from datasets import load_from_disk

dataset = load_from_disk('./data/ChnSentiCorp/')

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9600
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 0
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1200
    })
})

In [8]:
# 缩小数据规模, 便于测试
dataset['train'] = dataset['train'].shuffle().select(range(2000))
dataset['test'] = dataset['test'].shuffle().select(range(100))

In [9]:
dataset['train'][0]

{'text': '对比了好长时间才选择的这款，配置还是不错的，做工很精致，外观全黑很漂亮，送的赠品也不错，摄像头很清晰，.奥特蓝星音质也很好，性价比高',
 'label': 1}

In [10]:
def f(data, tokenizer):
    return tokenizer(data['text'], truncation=True)

dataset = dataset.map(f, batched=True,
                     batch_size=1000,
                     num_proc=4,
                     remove_columns=['text'],
                     fn_kwargs={'tokenizer': tokenizer})

dataset

Map (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['label'],
        num_rows: 0
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 100
    })
})

In [11]:
# 删掉太长的句子
def f(data):
    return [len(i) <= 512 for i in data['input_ids']]

dataset = dataset.filter(f, batched=True, batch_size=1000, num_proc=4)

Filter (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

In [12]:
from transformers import AutoModelForSequenceClassification

In [13]:
model = AutoModelForSequenceClassification.from_pretrained('hfl/rbt3', num_labels=2)

pytorch_model.bin:   0%|          | 0.00/156M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: hfl/rbt3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider traini

In [14]:
# 统计模型参数量
sum([i.nelement() for i in model.parameters()])

38478338

In [15]:
# 模型试算
import torch

# 模拟一条数据
data = {
    'input_ids': torch.ones(4, 10, dtype=torch.long),
    'token_type_ids': torch.ones(4, 10, dtype=torch.long),
    'attention_mask': torch.ones(4, 10, dtype=torch.long),
    'labels': torch.ones(4, dtype=torch.long)
}

out = model(**data)

In [16]:
out

SequenceClassifierOutput(loss=tensor(0.6901, grad_fn=<NllLossBackward0>), logits=tensor([[-0.0946, -0.0885],
        [-0.0946, -0.0885],
        [-0.0946, -0.0885],
        [-0.0946, -0.0885]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [17]:
out['loss']

tensor(0.6901, grad_fn=<NllLossBackward0>)

In [18]:
out['logits']

tensor([[-0.0946, -0.0885],
        [-0.0946, -0.0885],
        [-0.0946, -0.0885],
        [-0.0946, -0.0885]], grad_fn=<AddmmBackward0>)